<a href="https://colab.research.google.com/github/mbithi002/generativeai/blob/main/mbithi002_research_logs_vectorization_techniques.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# VECTORIZATION TECHNIQUES COMPARISON PROJECT

In [1]:
# 1. install requirements
!pip install gensim matplotlib seaborn scikit-learn plotly

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 53.2 MB/s eta 0:00:00


In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.decomposition import PCA
from sklearn.datasets import fetch_20newsgroups
import plotly.express as px
import plotly.graph_objects as go
from gensim.models import Word2Vec
from gensim.utils import simple_preprocess
import warnings
warnings.filterwarnings('ignore')

 LOAD DATASET (Kaggle's BBC News Dataset)

In [8]:
print("📥 Loading dataset...")

# Load subset of 20 newsgroups (smaller categories for faster processing)
categories = ['rec.sport.baseball', 'sci.space', 'comp.graphics', 'talk.politics.misc']
newsgroups = fetch_20newsgroups(subset='train', categories=categories, remove=('headers', 'footers', 'quotes'))

📥 Loading dataset...


In [9]:
df = pd.DataFrame({
    'Text': newsgroups.data,
    'Category': [newsgroups.target_names[i] for i in newsgroups.target]
})

print(f"✅ Dataset loaded: {df.shape[0]} articles")
print(f"Categories: {df['Category'].unique()}")

✅ Dataset loaded: 2239 articles
Categories: ['rec.sport.baseball' 'talk.politics.misc' 'sci.space' 'comp.graphics']


In [39]:
df.shape

(2239, 2)

In [41]:
# Sample 50 articles for demonstration
df_sample = df
print(f"📝 Using {len(df_sample)} sample articles")

📝 Using 2239 sample articles


In [42]:
# Preview
print("\n📋 Sample data:")
for i in range(3):
    print(f"\nCategory: {df_sample['Category'][i]}")
    print(f"Text: {df_sample['Text'][i][:150]}...")


📋 Sample data:

Category: rec.sport.baseball
Text: I thought that was Sandy Koufax....

Category: talk.politics.misc
Text: 
And the religious right worships engines, smokestacks, landfills,
and hates people.

What does this name-calling have to do with anything you are cla...

Category: talk.politics.misc
Text: 
How can a witness tell that someone in a burning truck is dead rather than
unconscious?


What kind of witnesses?  If we are talking about witnesses ...


TEXT PREPROCESSING

In [43]:
def preprocess_text(text):
    """Simple preprocessing for demonstration"""
    if isinstance(text, str):
        return text.lower()
    return ""

df_sample['clean_text'] = df_sample['Text'].apply(preprocess_text)

# TECHNIQUE 1: BAG OF WORDS (BoW)

In [44]:

print("TECHNIQUE 1: BAG OF WORDS (BoW) \n")

# Create BoW vectors
bow_vectorizer = CountVectorizer(max_features=50, stop_words='english')
bow_vectors = bow_vectorizer.fit_transform(df_sample['clean_text']).toarray()

print(f"BoW matrix shape: {bow_vectors.shape}")
print(f"Features (words): {bow_vectorizer.get_feature_names_out()[:15]}...")

TECHNIQUE 1: BAG OF WORDS (BoW) 

BoW matrix shape: (2239, 50)
Features (words): ['10' 'available' 'better' 'data' 'did' 'does' 'don' 'edu' 'file' 'going'
 'good' 'government' 'graphics' 'image' 'images']...


In [45]:
# Convert to DataFrame for inspection
bow_df = pd.DataFrame(
    bow_vectors,
    columns=bow_vectorizer.get_feature_names_out()
)
print("\n📊 BoW Sample (first 5 documents, first 10 words):")
print(bow_df.iloc[:5, :10])


📊 BoW Sample (first 5 documents, first 10 words):
   10  available  better  data  did  does  don  edu  file  going
0   0          0       0     0    0     0    0    0     0      0
1   0          1       0     0    0     1    0    0     0      0
2   0          0       0     1    0     0    1    0     0      0
3   0          0       0     0    0     0    0    0     0      0
4   0          0       0     0    0     0    0    1     0      0


# TECHNIQUE 2: TF-IDF

In [46]:
print("TECHNIQUE 2: TF-IDF \n")
# Create TF-IDF vectors
tfidf_vectorizer = TfidfVectorizer(max_features=50, stop_words='english')
tfidf_vectors = tfidf_vectorizer.fit_transform(df_sample['clean_text']).toarray()

tfidf_df = pd.DataFrame(
    tfidf_vectors,
    columns=tfidf_vectorizer.get_feature_names_out()
)
print(f"sample data {df_sample[:5]} \n")

print("\n📊 TF-IDF Sample (first 5 documents, first 10 words):")
print(tfidf_df.iloc[:5, :10])


print(f"TF-IDF matrix shape: {tfidf_vectors.shape}")

TECHNIQUE 2: TF-IDF 

sample data                                                 Text            Category  \
0                   I thought that was Sandy Koufax.  rec.sport.baseball   
1  \nAnd the religious right worships engines, sm...  talk.politics.misc   
2  \nHow can a witness tell that someone in a bur...  talk.politics.misc   
3  \n\n\n\nYes, long before Star Trek.  Before Ei...           sci.space   
4  \n\nIt depends.  If you can get your old veter...  rec.sport.baseball   

                                          clean_text  
0                   i thought that was sandy koufax.  
1  \nand the religious right worships engines, sm...  
2  \nhow can a witness tell that someone in a bur...  
3  \n\n\n\nyes, long before star trek.  before ei...  
4  \n\nit depends.  if you can get your old veter...   


📊 TF-IDF Sample (first 5 documents, first 10 words):
    10  available  better      data  did      does       don       edu  file  \
0  0.0   0.000000     0.0  0.000000  0.0  0

In [47]:
# Compare BoW vs TF-IDF for a specific document
doc_idx = 0
print(f"\n📊 Comparison for Document {doc_idx}:")
print(f"Category: {df_sample['Category'][doc_idx]}")
print(f"Original text: {df_sample['clean_text'][doc_idx][:100]}...\n")

# Show top words by BoW
bow_doc = bow_vectors[doc_idx]
top_bow_idx = bow_doc.argsort()[-5:][::-1]
print("Top BoW words (raw counts):")
for idx in top_bow_idx:
    if bow_doc[idx] > 0:
        word = bow_vectorizer.get_feature_names_out()[idx]
        print(f"  - {word}: {int(bow_doc[idx])}")

# Show top words by TF-IDF
tfidf_doc = tfidf_vectors[doc_idx]
top_tfidf_idx = tfidf_doc.argsort()[-5:][::-1]
print("\nTop TF-IDF words (weighted importance):")
for idx in top_tfidf_idx:
    if tfidf_doc[idx] > 0:
        word = tfidf_vectorizer.get_feature_names_out()[idx]
        print(f"  - {word}: {tfidf_doc[idx]:.3f}")


📊 Comparison for Document 0:
Category: rec.sport.baseball
Original text: i thought that was sandy koufax....

Top BoW words (raw counts):

Top TF-IDF words (weighted importance):


# TECHNIQUE 3: WORD2VEC (Document Vectors)

In [48]:
print("TECHNIQUE 3: WORD2VEC \n")

# Tokenize documents for Word2Vec
tokenized_docs = [simple_preprocess(doc) for doc in df_sample['clean_text']]

# Train Word2Vec model
print("Training Word2Vec model...")
w2v_model = Word2Vec(
    sentences=tokenized_docs,
    vector_size=50,  # 50-dimensional vectors
    window=5,
    min_count=1,
    workers=4,
    epochs=10
)
print("✅ Word2Vec model trained!")

TECHNIQUE 3: WORD2VEC 

Training Word2Vec model...
✅ Word2Vec model trained!


In [49]:
# Create document vectors by averaging word vectors
def document_vector(doc_tokens, model):
    """Create document vector by averaging word vectors"""
    valid_tokens = [token for token in doc_tokens if token in model.wv]
    if not valid_tokens:
        return np.zeros(model.vector_size)
    return np.mean(model.wv[valid_tokens], axis=0)

# Generate document vectors
w2v_doc_vectors = np.array([document_vector(doc, w2v_model) for doc in tokenized_docs])
print(f"Word2Vec document vectors shape: {w2v_doc_vectors.shape}")

# Show word similarities
print("\n📊 Word Similarities:")
word_pairs = [
    ('space', 'moon'),
    ('baseball', 'game'),
    ('computer', 'graphics'),
    ('government', 'political')
]

for word1, word2 in word_pairs:
    if word1 in w2v_model.wv and word2 in w2v_model.wv:
        similarity = w2v_model.wv.similarity(word1, word2)
        print(f"  Similarity between '{word1}' and '{word2}': {similarity:.3f}")
    else:
        missing = []
        if word1 not in w2v_model.wv: missing.append(word1)
        if word2 not in w2v_model.wv: missing.append(word2)
        print(f"  Words not in vocabulary: {missing}")

Word2Vec document vectors shape: (2239, 50)

📊 Word Similarities:
  Similarity between 'space' and 'moon': 0.486
  Similarity between 'baseball' and 'game': 0.835
  Similarity between 'computer' and 'graphics': 0.857
  Similarity between 'government' and 'political': 0.731


# VISUALIZATION: 3D COMPARISON

In [51]:
print("\n" + "="*60)
print("3D VISUALIZATION")
print("="*60)

from sklearn.decomposition import PCA
import plotly.graph_objects as go
import numpy as np

# Reduce dimensions to 3D
def reduce_to_3d(vectors):
    reducer = PCA(n_components=3, random_state=42)
    return reducer.fit_transform(vectors)

# Reduce vectors
bow_3d = reduce_to_3d(bow_vectors)
tfidf_3d = reduce_to_3d(tfidf_vectors)
w2v_3d = reduce_to_3d(w2v_doc_vectors)

# Simplify category names
df_sample['simple_cat'] = df_sample['Category'].apply(lambda x: x.split('.')[-1])
categories = df_sample['simple_cat'].values
unique_categories = np.unique(categories)

# Function to create 3D plot
def create_3d_plot(vectors_3d, title):
    fig = go.Figure()

    for cat in unique_categories:
        idx = categories == cat

        fig.add_trace(go.Scatter3d(
            x=vectors_3d[idx, 0],
            y=vectors_3d[idx, 1],
            z=vectors_3d[idx, 2],
            mode='markers',
            marker=dict(size=4, opacity=0.7),
            name=cat
        ))

    fig.update_layout(
        title=title,
        scene=dict(
            xaxis_title='PC1',
            yaxis_title='PC2',
            zaxis_title='PC3'
        ),
        width=800,
        height=600,
        showlegend=True
    )

    fig.show()


3D VISUALIZATION


In [52]:
# Create separate visualizations
create_3d_plot(bow_3d, "BoW - 3D PCA Visualization")

In [53]:
create_3d_plot(tfidf_3d, "TF-IDF - 3D PCA Visualization")

In [54]:

create_3d_plot(w2v_3d, "Word2Vec - 3D PCA Visualization")

# SIMILARITY DETECTION VISUALIZATION

In [59]:
from sklearn.metrics.pairwise import cosine_similarity

for method_name, vectors in methods.items():
    print(f"\n{method_name}:")

    sims = cosine_similarity(
        vectors[query_idx].reshape(1, -1),
        vectors
    )[0]

    similar_indices = np.argsort(sims)[::-1][1:4]

    for idx in similar_indices:
        sim_score = sims[idx]
        category = df_sample['simple_cat'][idx]
        text_preview = df_sample['clean_text'][idx][:50]
        print(f"  → {category}: {sim_score:.3f} - '{text_preview}...'")


BoW:
  → graphics: 0.000 - 'g'day all,

can anybody point me at a utility whic...'
  → misc: 0.000 - 'the white house


                  office of the ...'
  → space: 0.000 - '
there was a recession, and none of the potential ...'

TF-IDF:
  → graphics: 0.000 - 'g'day all,

can anybody point me at a utility whic...'
  → misc: 0.000 - 'the white house


                  office of the ...'
  → space: 0.000 - '
there was a recession, and none of the potential ...'

Word2Vec:
  → baseball: 0.949 - 'i thought that walt weiss was jewish.  i seem to r...'
  → baseball: 0.929 - '



that last was me, steve novak.  i've since rea...'
  → space: 0.929 - '                                     ^^^^^^^^^^^^^...'


# CATEGORY CLUSTERING VISUALIZATION

In [60]:
print("\n" + "="*60)
print("CATEGORY CLUSTERING")
print("="*60)

# Color by category
colors = {'baseball': 'red', 'space': 'blue', 'graphics': 'green', 'politics': 'orange'}
point_colors = [colors.get(cat, 'gray') for cat in df_sample['simple_cat']]

# Create 3D plot colored by category
fig = go.Figure()

for method_name, vectors_3d, color_prefix in [
    ('BoW', bow_3d, 'red'),
    ('TF-IDF', tfidf_3d, 'blue'),
    ('Word2Vec', w2v_3d, 'green')
]:
    for category in df_sample['simple_cat'].unique():
        mask = df_sample['simple_cat'] == category
        fig.add_trace(go.Scatter3d(
            x=vectors_3d[mask, 0],
            y=vectors_3d[mask, 1],
            z=vectors_3d[mask, 2],
            mode='markers',
            marker=dict(size=4),
            name=f'{method_name}-{category}',
            legendgroup=category,
            showlegend=True if method_name == 'BoW' else False
        ))

fig.update_layout(
    title="Document Clusters by Category",
    scene=dict(
        xaxis_title='PC1',
        yaxis_title='PC2',
        zaxis_title='PC3'
    ),
    width=800,
    height=600
)

fig.show()


CATEGORY CLUSTERING
